In [1]:
from getpass import getpass
from qiskit_ibm_runtime import QiskitRuntimeService

API_KEY = getpass("Enter your IBM Quantum API key: ")
CRN = getpass("Enter your IBM Quantum CRN: ")

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=API_KEY,
    instance=CRN
)

print("Connected to IBM Quantum successfully!")

Enter your IBM Quantum API key:  ········
Enter your IBM Quantum CRN:  ········


qiskit_runtime_service._discover_account:WARNING:2026-10-12 13:26:58,662: Loading account with the given token. A saved account will not be used.


Connected to IBM Quantum successfully!


In [2]:
from qiskit import QuantumCircuit

# Original Bell-state circuit
original = QuantumCircuit(2, 2)
original.h(0)
original.cx(0, 1)
original.measure([0, 1], [0, 1])

# Modified Bell-state circuit
modified = QuantumCircuit(2, 2)
modified.h(0)
modified.cx(0, 1)
modified.x(1)  # Add one X gate
modified.measure([0, 1], [0, 1])

print("Original Bell Circuit:")
print(original.draw())

print("\nModified Bell Circuit:")
print(modified.draw())


Original Bell Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

Modified Bell Circuit:
     ┌───┐          ┌─┐   
q_0: ┤ H ├──■───────┤M├───
     └───┘┌─┴─┐┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


In [ ]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

# Find an available real quantum processor
backend = service.least_busy(
    operational=True,
    simulator=False
)

print("Selected quantum processor:", backend.name)

# Compile the circuits for the selected processor
pass_manager = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1
)

isa_original = pass_manager.run(original)
isa_modified = pass_manager.run(modified)

# Execute each circuit for 100 shots
sampler = SamplerV2(mode=backend)

job = sampler.run(
    [isa_original, isa_modified],
    shots=100
)

print("Job ID:", job.job_id())

# Retrieve results
result = job.result()

counts_original = result[0].data.c.get_counts()
counts_modified = result[1].data.c.get_counts()

print("\nOriginal circuit results:", counts_original)
print("Modified circuit results:", counts_modified)

Selected quantum processor: ibm_kingston


C:\Users\admin\AppData\Local\Temp\ipykernel_4400\1829231744.py:22: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = SamplerV2(mode=backend)


Job ID: db695samb58s738bet5g


In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

plot_histogram(
    [counts_original, counts_modified],
    legend=["Original Bell Circuit", "Modified Bell Circuit"],
    title="Challenge 3: Change the Correlation"
)

plt.show()

In [ ]:
def show_results(name, counts):
    total = sum(counts.values())

    print(f"\n{name}")
    print("-" * 35)

    for outcome in ["00", "01", "10", "11"]:
        count = counts.get(outcome, 0)
        percentage = count / total * 100
        print(f"{outcome}: {count} shots ({percentage:.1f}%)")

show_results("Original Circuit", counts_original)
show_results("Modified Circuit", counts_modified)